### Tutorial 3: Training a WSI Classification Model with ABMIL

This tutorial will guide you step-by-step to train an attention-based multiple instance learning model using Trident patch embeddings. Then, we will generation attention heatmaps using the pretrained model. 


#### A- Installation and patch feature extraction using UNI

#### Step 1: Download a dataset of whole-slide images

You can use your own WSIs or download a publicly available dataset, e.g. from:

- **CPTAC CCRCC WSIs**: Download from the [TCIA Cancer Imaging Archive](https://www.cancerimagingarchive.net/collection/cptac-ccrcc/).
- **Store WSIs**: Save all WSIs into a local directory, e.g.,  
  ```bash
  ./CPTAC-CCRCC_v1/CCRCC
  ```

#### Step 2:  Run CONCH v1.5 feature extraction:

Navigate to the base directory of Trident and execute the following command:

```bash
python run_batch_of_slides.py --task all \
  --wsi_dir ./CPTAC-CCRCC_v1/CCRCC \
  --job_dir ./tutorial-3 \
  --patch_encoder conch_v15 \
  --mag 20 \
  --patch_size 512
```


#### B- Download labels with data splits

Here, we use Patho-Bench CPTAC-CCRCC labels for predicting BAP1 mutation. 

In [3]:
!pip install datasets

  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 11.8 MB/s  0:00:00
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.5 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [datasets]4/5 [datasets]ess]


In [ ]:
import datasets
import pandas as pd

# # Download labels as csv
# datasets.load_dataset(
#     'MahmoodLab/Patho-Bench', 
#     cache_dir='./tutorial-3',
#     dataset_to_download='cptac_ccrcc',     
#     task_in_dataset='BAP1_mutation',           
#     trust_remote_code=True
# )

# # Visualize my labels and splits
# df = pd.read_csv('tutorial-3/cptac_ccrcc/BAP1_mutation/k=all.tsv', sep="\t")
# df

# Visualize my labels and splits

,patient,type,fold_0,fold_1,fold_2,fold_3,fold_4
199,TCGA-D5-6927,1,train,train,train,train,train
200,TCGA-D5-6928,1,train,train,val,train,train
201,TCGA-D5-6930,1,test,train,train,val,train


In [2]:
# # Check the label distribution
# df_counts = df['BAP1_mutation'].value_counts().reset_index()
# df_counts.columns = ['BAP1_mutation', 'Count']
# df_counts

df_counts = df['patient'].value_counts().reset_index()
df_counts.columns = ['patient', 'Count']
df_counts



,patient,Count
0,TCGA-3L-AA1B,1
1,TCGA-4N-A93T,1
2,TCGA-5M-AAT4,1
3,TCGA-5M-AAT6,1
4,TCGA-5M-AATE,1
...,...,...
261,TCGA-QG-A5YV,1
262,TCGA-QG-A5Z2,1
263,TCGA-QL-A97D,1
264,TCGA-SS-A7HO,1


#### C- Training an ABMIL model

In [6]:
import sys
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import h5py
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

sys.path.append(os.path.join(os.getcwd(), 'TRIDENT'))

from trident.slide_encoder_models import ABMILSlideEncoder

# Set deterministic behavior
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


class BinaryClassificationModel(nn.Module):
    def __init__(self, input_feature_dim=768, n_heads=1, head_dim=512, dropout=0., gated=True, hidden_dim=256):
        super().__init__()
        self.feature_encoder = ABMILSlideEncoder(
            freeze=False,
            input_feature_dim=input_feature_dim, 
            n_heads=n_heads, 
            head_dim=head_dim, 
            dropout=dropout, 
            gated=gated
        )
        self.classifier = nn.Sequential(
            nn.Linear(input_feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, return_raw_attention=False):
        if return_raw_attention:
            features, attn = self.feature_encoder(x, return_raw_attention=True)
        else:
            features = self.feature_encoder(x)
        logits = self.classifier(features).squeeze(1)
        
        if return_raw_attention:
            return logits, attn
        
        return logits

# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BinaryClassificationModel(input_feature_dim=1536).to(device)
class H5Dataset(Dataset):
    def __init__(self, feats_path, df, split, num_features=512):
        self.df = df[df["fold_0"] == split].reset_index(drop=True) # 인덱스 초기화 안전장치
        self.feats_path = feats_path
        self.num_features = num_features
        self.split = split
        
        # 1. 초기화 단계에서 환자별 파일 매핑 딕셔너리 생성 (속도 최적화)
        self.patient_to_files = {}
        all_files = os.listdir(feats_path)
        
        for p_id in self.df['patient']:
            # 환자 ID(12자리)로 시작하고 .h5로 끝나는 모든 파일 찾기
            matching_files = [
                os.path.join(feats_path, f) for f in all_files 
                if f.startswith(p_id) and f.endswith('.h5')
            ]
            self.patient_to_files[p_id] = matching_files

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        patient_id = row['patient']
        file_paths = self.patient_to_files[patient_id]
        
        if not file_paths:
            raise FileNotFoundError(f"환자 {patient_id}에 대한 .h5 파일을 찾을 수 없습니다.")

        # 2. 여러 슬라이드의 피처를 리스트에 담은 후 하나로 병합
        all_features = []
        for fp in file_paths:
            with h5py.File(fp, "r") as f:
                all_features.append(torch.from_numpy(f["features"][:]))
        
        # 차원 0(인스턴스 개수)을 기준으로 결합 (예: [5000, 1536] + [3000, 1536] = [8000, 1536])
        features = torch.cat(all_features, dim=0)

        # 3. 학습 시 고정된 개수로 샘플링 (기존 로직 유지)
        if self.split == 'train':
            num_available = features.shape[0]
            if num_available >= self.num_features:
                indices = torch.randperm(num_available, generator=torch.Generator().manual_seed(SEED))[:self.num_features]
            else:
                indices = torch.randint(num_available, (self.num_features,), generator=torch.Generator().manual_seed(SEED))  # Oversampling
            features = features[indices]

        label = torch.tensor(row["type"], dtype=torch.float32)
        return features, label
# Create dataloaders
feats_path = '/home/team1/data/trident_processed/20.0x_256px_0px_overlap/features_uni_v2'
batch_size = 8
train_loader = DataLoader(H5Dataset(feats_path, df, "train"), batch_size=batch_size, shuffle=True, worker_init_fn=lambda _: np.random.seed(SEED))
test_loader = DataLoader(H5Dataset(feats_path, df, "test"), batch_size=1, shuffle=False, worker_init_fn=lambda _: np.random.seed(SEED))

# Training setup
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=4e-4)

# Training loop
num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.
    for features, labels in train_loader:
        features, labels = {'features': features.to(device)}, labels.to(device)
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1/20, Loss: 0.4904
Epoch 2/20, Loss: 0.3059
Epoch 3/20, Loss: 0.3423
Epoch 4/20, Loss: 0.1646
Epoch 5/20, Loss: 0.1679
Epoch 6/20, Loss: 0.0462
Epoch 7/20, Loss: 0.0872
Epoch 8/20, Loss: 0.0574
Epoch 9/20, Loss: 0.0166
Epoch 10/20, Loss: 0.0096
Epoch 11/20, Loss: 0.0014
Epoch 12/20, Loss: 0.0008
Epoch 13/20, Loss: 0.0003
Epoch 14/20, Loss: 0.0002
Epoch 15/20, Loss: 0.0001
Epoch 16/20, Loss: 0.0001
Epoch 17/20, Loss: 0.0001
Epoch 18/20, Loss: 0.0001
Epoch 19/20, Loss: 0.0000
Epoch 20/20, Loss: 0.0001


#### D- Evaluating the ABMIL model

In [15]:
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
import torch

# Evaluation
model.eval()
all_labels, all_probs = [], []  # outputs 대신 probs(확률값) 저장으로 변경
correct = 0
total = 0

with torch.no_grad():
    for features, labels in test_loader:
        # TRIDENT 규격에 맞춘 딕셔너리 전달
        features, labels = {'features': features.to(device)}, labels.to(device)
        
        outputs = model(features) # 원본 로짓 (Raw logits)
        
        # 1. 로짓을 0~1 사이의 확률값으로 변환 (지식 증류 및 해석 용이)
        probs = torch.sigmoid(outputs)
        
        # 2. 정확도 계산 (로짓이 0보다 크거나, 확률이 0.5보다 크면 MSI(1)로 예측)
        predicted = (outputs > 0).float()  
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        all_probs.append(probs.cpu().numpy())  
        all_labels.append(labels.cpu().numpy())

# 결과 결합
all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)

# 3. 다양한 평가 지표 계산
auc = roc_auc_score(all_labels, all_probs)
auprc = average_precision_score(all_labels, all_probs) # AUPRC 추가
accuracy = correct / total

print(f"Test AUC: {auc:.4f}")
print(f"Test AUPRC: {auprc:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Test AUC: 0.8852
Test AUPRC: 0.8526
Test Accuracy: 0.9385


#### E- Extract attention heatmap for the freshly trained model

In [ ]:
from trident import OpenSlideWSI, visualize_heatmap
from trident.segmentation_models import segmentation_model_factory
from trident.patch_encoder_models import encoder_factory as patch_encoder_factory

# a. Load WSI to process
job_dir = './tutorial-3/heatmap_viz'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
slide = OpenSlideWSI(slide_path='./CPTAC-CCRCC_v1/CCRCC/C3L-00418-22.svs', lazy_init=False)

# b. Run segmentation 
segmentation_model = segmentation_model_factory("hest")
geojson_contours = slide.segment_tissue(segmentation_model=segmentation_model, job_dir=job_dir, device=device)

# c. Run patch coordinate extraction
coords_path = slide.extract_tissue_coords(
    target_mag=20,
    patch_size=512,
    save_coords=job_dir,
    overlap=256, 
)

# d. Run patch feature extraction
patch_encoder = patch_encoder_factory("conch_v15").eval().to(device)
patch_features_path = slide.extract_patch_features(
    patch_encoder=patch_encoder,
    coords_path=coords_path,
    save_features=os.path.join(job_dir, f"features_conch_v15"),
    device=device
)

#  e. Run inference 
with h5py.File(patch_features_path, 'r') as f:
    coords = f['coords'][:]
    patch_features = f['features'][:]
    coords_attrs = dict(f['coords'].attrs)

batch = {'features': torch.from_numpy(patch_features).float().to(device).unsqueeze(0)}
logits, attention = model(batch, return_raw_attention=True)

# f. generate heatmap
heatmap_save_path = visualize_heatmap(
    wsi=slide,
    scores=attention.cpu().numpy().squeeze(),  
    coords=coords,
    vis_level=1,
    patch_size_level0=coords_attrs['patch_size_level0'],
    normalize=True,
    num_top_patches_to_save=10,
    output_dir=job_dir
)

